**Project Objective**

This project aims to use demographic, body measurement, blood pressure, and lifestyle characteristics—such as age, BMI, waist circumference, smoking status, and physical activity—to estimate the likelihood of having an HbA1c result in the diabetes range among adults who have not previously reported a diabetes diagnosis.

The model is intended to support risk screening and research. It does not provide a clinical diagnosis.

(TR: Daha önce diyabet teşhisi almamış kişilerin yaş, BMI, bel çevresi, kan basıncı, ve yaşam tarzı bilgilerini kullanrak, HbA1c testlerinin diyabet aralığında çıkma ihtimalini tahmin etmeye çalışıyorum.) 

In [1]:
# Import necessary libraries
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

## Data Preparation

In [2]:
# Set the main Data path 
DATA_DIR = Path("../dataset")

# Read all the files
demo = pd.read_sas(DATA_DIR/"DEMO_L.xpt", format="xport")
diabetes = pd.read_sas(DATA_DIR/"DIQ_L.xpt", format="xport")
hba1c = pd.read_sas(DATA_DIR/"GHB_L.xpt", format="xport")
body = pd.read_sas(DATA_DIR / "BMX_L.xpt", format="xport")

In [3]:
# Inspect the files 
datasets = {
    "demographics": demo,
    "diabetes": diabetes,
    "hba1c": hba1c,
    "body_measurements": body
}

# Display the dataset names with their rows and columns information
for dataset_name, frame in datasets.items():
    print(f"- {dataset_name.capitalize()} \n\t rows: {frame.shape[0]} \n\t columns: {frame.shape[1]} ")

- Demographics 
	 rows: 11933 
	 columns: 27 
- Diabetes 
	 rows: 11744 
	 columns: 9 
- Hba1c 
	 rows: 7199 
	 columns: 3 
- Body_measurements 
	 rows: 8860 
	 columns: 22 


There are different number of rows exist in each file, it's showing that not everybody participated in the tests, forms, or examinations. 

In [4]:
for name, frame in datasets.items():
    print(f"- {name.capitalize()} \n\t unique SEQN: {frame['SEQN'].nunique()} \n\t duplicate SEQN: {frame['SEQN'].duplicated().sum()} ")

- Demographics 
	 unique SEQN: 11933 
	 duplicate SEQN: 0 
- Diabetes 
	 unique SEQN: 11744 
	 duplicate SEQN: 0 
- Hba1c 
	 unique SEQN: 7199 
	 duplicate SEQN: 0 
- Body_measurements 
	 unique SEQN: 8860 
	 duplicate SEQN: 0 


SEQN variable is a unique identifier of the participant/patient.

In [5]:
# Merge the files into one file
merged_df = (
    demo
    .merge(diabetes,
               on="SEQN",
               how="left",
               validate="one_to_one"
    )
    .merge(hba1c,
               on="SEQN",
               how="left",
               validate="one_to_one"
    )
    .merge(body,
               on="SEQN",
               how="left",
               validate="one_to_one"
    )
)

The reason why we are performing left merge is because our main participants/patients are in the demo list. We want to preserve all demographic participants and add the other data files into it. 

In [6]:
merged_df.shape

(11933, 58)

In [7]:
merged_df.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,BMXLEG,BMILEG,BMXARML,BMIARML,BMXARMC,BMIARMC,BMXWAIST,BMIWAIST,BMXHIP,BMIHIP
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,42.8,NaN,42.0,NaN,35.7,NaN,98.3,NaN,102.9,NaN
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,38.5,NaN,38.7,NaN,33.7,NaN,114.7,NaN,112.4,NaN
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,38.5,NaN,35.5,NaN,36.3,NaN,93.5,NaN,98.0,NaN
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,NaN,NaN,25.4,NaN,23.4,NaN,70.4,NaN,NaN,NaN
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,NaN,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,NaN


In [8]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11933 entries, 0 to 11932
Data columns (total 58 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      11933 non-null  float64
 1   SDDSRVYR  11933 non-null  float64
 2   RIDSTATR  11933 non-null  float64
 3   RIAGENDR  11933 non-null  float64
 4   RIDAGEYR  11933 non-null  float64
 5   RIDAGEMN  377 non-null    float64
 6   RIDRETH1  11933 non-null  float64
 7   RIDRETH3  11933 non-null  float64
 8   RIDEXMON  8860 non-null   float64
 9   RIDEXAGM  2787 non-null   float64
 10  DMQMILIZ  8301 non-null   float64
 11  DMDBORN4  11914 non-null  float64
 12  DMDYRUSR  1875 non-null   float64
 13  DMDEDUC2  7794 non-null   float64
 14  DMDMARTZ  7792 non-null   float64
 15  RIDEXPRG  1503 non-null   float64
 16  DMDHHSIZ  11933 non-null  float64
 17  DMDHRGND  4115 non-null   float64
 18  DMDHRAGZ  4124 non-null   float64
 19  DMDHREDZ  3746 non-null   float64
 20  DMDHRMAZ  4020 non-null   float64
 21  

In [9]:
merged_df["RIDAGEYR"] >= 18

0         True
1         True
2         True
3        False
4        False
         ...  
11928    False
11929     True
11930     True
11931     True
11932     True
Name: RIDAGEYR, Length: 11933, dtype: bool

In [10]:
# Create a cohort of 
    # adults older than 18 years old
    # no previous diabetes diagnosed/reported
    # not pregnant
    # does have HbA1c test result

is_adult = merged_df["RIDAGEYR"] >= 18
reports_no_diabetes = merged_df["DIQ010"] == 2
has_hba1c = merged_df["LBXGH"].notna()
not_pregnant = (
    merged_df["RIDEXPRG"].isna() | 
    (merged_df["RIDEXPRG"] != 1)
)


In [11]:
# Merge all the conditions in a copy of dataframe 
cohort_df = merged_df[
    is_adult
    & reports_no_diabetes
    & has_hba1c
    & not_pregnant
].copy()

In [12]:
cohort_df.shape

(4929, 58)

There are 4929 individuals in this specific cohort we'll be analyzing.

In [13]:
# Create the target variable
cohort_df["a1c_diabetes_range"] = (
    cohort_df["LBXGH"] >= 6.5
).astype(int)

cohort_df["a1c_diabetes_range"].value_counts()

a1c_diabetes_range
0    4813
1     116
Name: count, dtype: int64

0 → HbA1c < 6.5% <br>
1 → HbA1c ≥ 6.5%

In [14]:
cohort_df["a1c_diabetes_range"].value_counts(normalize=True) * 100

a1c_diabetes_range
0    97.646581
1     2.353419
Name: proportion, dtype: float64

The positive test result is only about 2.35% which indicates that the data is imbalanced. We have to be careful about the accuracy metric because even if model always predicts 0 it could achieve %97.65 accuracy, so we won't use accuracy as our primary evaulation metric. 

I'll use the features below to create the first version of the model. Then, we can add more features and test if they affect the model performance in someway.

In [15]:
features = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "INDFMPIR",
    "BMXBMI",
    "BMXWAIST"
]

target = "a1c_diabetes_range"

In [16]:
feature_summary = (
    cohort_df[features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

feature_summary

INDFMPIR    12.822073
DMDEDUC2     4.686549
BMXWAIST     4.017042
BMXBMI       1.278150
RIDAGEYR     0.000000
RIAGENDR     0.000000
RIDRETH3     0.000000
dtype: float64

These table shows percentage of each features' missing data. 

In [17]:
analysis_df = cohort_df[
    features + [target]
].copy()

analysis_df.head()

,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,a1c_diabetes_range
0,43.0,1.0,6.0,5.0,5.00,27.0,98.3,0
1,66.0,1.0,3.0,5.0,5.00,33.5,114.7,0
8,34.0,1.0,1.0,4.0,1.33,30.2,106.1,0
9,68.0,2.0,3.0,5.0,1.32,42.6,122.0,0
10,27.0,2.0,4.0,4.0,0.81,43.7,118.5,0


In [18]:
analysis_df.shape

(4929, 8)

In [19]:
analysis_df.info()

<class 'pandas.DataFrame'>
Index: 4929 entries, 0 to 11932
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RIDAGEYR            4929 non-null   float64
 1   RIAGENDR            4929 non-null   float64
 2   RIDRETH3            4929 non-null   float64
 3   DMDEDUC2            4698 non-null   float64
 4   INDFMPIR            4297 non-null   float64
 5   BMXBMI              4866 non-null   float64
 6   BMXWAIST            4731 non-null   float64
 7   a1c_diabetes_range  4929 non-null   int64  
dtypes: float64(7), int64(1)
memory usage: 346.6 KB


DMDEDUC2 only exists for those who are older than 20 years old, since our cohort also includes those who are 18 and 19 years old, we need to decide whether to fill out these data during preprocessing phase or have the cohort include those only older than 20 years old.

In [20]:
# Check the sizes of each group
print(
    "Age 18+:",
    (cohort_df["RIDAGEYR"] >= 18).sum()
)

print(
    "Age 20+:",
    (cohort_df["RIDAGEYR"] >= 20).sum()
)

Age 18+: 4929
Age 20+: 4698


In [21]:
cohort_df.groupby(
    cohort_df["RIDAGEYR"] >= 20
)["a1c_diabetes_range"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
RIDAGEYR,,,
False,231,0,0.000000
True,4698,116,0.024691


| Group      |  Person | Positive | Positive Percentage |
| --------- | ----: | ------: | -----------: |
| 18–19 age |   231 |       0 |           %0 |
| 20+ age   | 4.698 |     116 |        %2,47 |


The initial cohort included adults aged 18 years and older. However, the `DMDEDUC2` education variable is only available for participants aged 20 years and older.

The 18–19-year-old group contained 231 participants and no diabetes-range HbA1c outcomes. Therefore, the minimum eligibility age was changed from 18 to 20 years to maintain a consistent adult cohort and ensure that education information was applicable to all eligible participants.

The absence of positive outcomes among participants aged 18–19 in this sample should not be interpreted as evidence that this age group has no diabetes risk.

In [22]:
is_adult = merged_df["RIDAGEYR"] >= 20
reports_no_diabetes = merged_df["DIQ010"] == 2
has_hba1c = merged_df["LBXGH"].notna()

not_pregnant = (
    merged_df["RIDEXPRG"].isna()
    | (merged_df["RIDEXPRG"] != 1)
)

cohort_df = merged_df[
    is_adult
    & reports_no_diabetes
    & has_hba1c
    & not_pregnant
].copy()

cohort_df["a1c_diabetes_range"] = (
    cohort_df["LBXGH"] >= 6.5
).astype(int)

In [23]:
print(cohort_df.shape)

print(
    cohort_df["a1c_diabetes_range"]
    .value_counts()
)

print(
    cohort_df["a1c_diabetes_range"]
    .value_counts(normalize=True) * 100
)

(4698, 59)
a1c_diabetes_range
0    4582
1     116
Name: count, dtype: int64
a1c_diabetes_range
0    97.530864
1     2.469136
Name: proportion, dtype: float64


In [24]:
# Create the analysis table after updating the cohort
analysis_df = cohort_df[
    [
        "RIDAGEYR",
        "RIAGENDR",
        "RIDRETH3",
        "DMDEDUC2",
        "INDFMPIR",
        "BMXBMI",
        "BMXWAIST",
        "a1c_diabetes_range"
    ]
].copy()

In [25]:
# Analyze the missing values
missing_summary = pd.DataFrame({
    "missing_count": analysis_df.isnull().sum(),
    "missing_percent": (
        analysis_df.isna().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "missing_percent",
    ascending=False
)


,missing_count,missing_percent
INDFMPIR,601,12.79
BMXWAIST,186,3.96
BMXBMI,57,1.21
RIDAGEYR,0,0.00
RIAGENDR,0,0.00
RIDRETH3,0,0.00
DMDEDUC2,0,0.00
a1c_diabetes_range,0,0.00


In [26]:
analysis_df['DMDEDUC2'].value_counts()

DMDEDUC2
5.0    1834
4.0    1432
3.0     944
2.0     307
1.0     179
9.0       2
Name: count, dtype: int64

**Meaning of each value:** <br>
1: Less than 9th grade <br>
2: 9th to 11th grade (including 12th grade with no diploma) <br>
3: High school graduate, GED, or equivalent <br>
4: Some college or AA degree <br> 
5: College graduate or above <br>
7: Refused <br>
9: Don't know <br>

It's better to replace Don't know's with NaN since it doesn't represent anything. 

In [27]:
analysis_columns = [
    # Identifier
    "SEQN",

    # Seven candidate features
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "INDFMPIR",
    "BMXBMI",
    "BMXWAIST",

    # Target source
    "LBXGH",

    # Survey variables 
    "WTPH2YR",
    "SDMVSTRA",
    "SDMVPSU",

    # Weight
    "WTMEC2YR",
    # Target
    "a1c_diabetes_range"
]

analysis_df = cohort_df[analysis_columns].copy()
# Recode invalid education responses as missing
analysis_df["DMDEDUC2"] = analysis_df["DMDEDUC2"].replace({
    7: np.nan,
    9: np.nan
})

For further analysis in EDA phase, I restored some of the features.

In [28]:
analysis_df["DMDEDUC2"].value_counts(
    dropna=False
).sort_index()

DMDEDUC2
1.0     179
2.0     307
3.0     944
4.0    1432
5.0    1834
NaN       2
Name: count, dtype: int64

In [29]:
missing_summary = pd.DataFrame({
    "missing_count": analysis_df.isna().sum(),
    "missing_percent": (
        analysis_df.isna().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

,missing_count,missing_percent
INDFMPIR,601,12.79
BMXWAIST,186,3.96
BMXBMI,57,1.21
DMDEDUC2,2,0.04
SEQN,0,0.00
RIDAGEYR,0,0.00
RIAGENDR,0,0.00
RIDRETH3,0,0.00
LBXGH,0,0.00
WTPH2YR,0,0.00


In [30]:
# Save the data
OUTPUT_DIR = Path("../artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

In [31]:
output_path = OUTPUT_DIR / "analysis_cohort.csv"

analysis_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")
print(f"Shape: {analysis_df.shape}")

Saved to: ../artifacts/analysis_cohort.csv
Shape: (4698, 14)


In [32]:
print(analysis_df.columns.tolist())

['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR', 'BMXBMI', 'BMXWAIST', 'LBXGH', 'WTPH2YR', 'SDMVSTRA', 'SDMVPSU', 'WTMEC2YR', 'a1c_diabetes_range']


## Validation: Outcome Prevalence Check

Before finalizing the prepared dataset, we validate our outcome definition 
(`a1c_diabetes_range`) by comparing its weighted prevalence against NCHS's 
published estimate for this cycle.

In [33]:
def weighted_prevalence(df, outcome_col, weight_col):
    w = df[weight_col]
    y = df[outcome_col]
    return np.sum(w * y) / np.sum(w)

unweighted_prev = analysis_df["a1c_diabetes_range"].mean()
weighted_prev = weighted_prevalence(analysis_df, "a1c_diabetes_range", "WTMEC2YR")

print(f"Unweighted prevalence: {unweighted_prev:.3%}")
print(f"Weighted prevalence:   {weighted_prev:.3%}")
print("CDC published estimate (Aug 2021-Aug 2023, HbA1c + fasting glucose): 4.5%")

Unweighted prevalence: 2.469%
Weighted prevalence:   2.026%
CDC published estimate (Aug 2021-Aug 2023, HbA1c + fasting glucose): 4.5%


Our weighted prevalence estimate (2.026%) is lower than NCHS's published estimate 
of 4.5% for this cycle. This is expected: our outcome definition uses only the 
HbA1c ≥ 6.5% criterion, while CDC's official undiagnosed diabetes definition also 
includes fasting plasma glucose ≥ 126 mg/dL. Since these two criteria do not 
perfectly overlap in the population, an HbA1c-only definition captures a subset 
of true undiagnosed cases. This is a deliberate scope decision made to avoid 
the added complexity of the separate fasting-subsample weight required for 
GLU_L — and is discussed further in the Limitations section.

In [34]:
# Test Check
assert analysis_df["WTMEC2YR"].notna().all()
assert (analysis_df["WTMEC2YR"] > 0).all()
assert analysis_df["SEQN"].is_unique
assert analysis_df["LBXGH"].notna().all()
assert analysis_df["RIDAGEYR"].min() >= 20
assert set(
    analysis_df["a1c_diabetes_range"].unique()
) <= {0, 1}

print("All preparation checks passed.")

All preparation checks passed.
